# Phase 3 RAG Evaluation

This notebook checks whether the knowledge-base retriever returns the right finding for simple clinical questions.

It uses the Chroma index built in `03_rag.ipynb`. It does not rebuild the knowledge base.

## What This Notebook Must Prove

- The saved Chroma index can be loaded.
- Queries retrieve the correct finding near the top.
- We can measure retrieval with recall@k, precision@k, and MRR.
- Out-of-scope questions get weaker scores than clear in-scope questions.

In [14]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CHUNKS_PATH = PROJECT_ROOT / "data" / "kb" / "processed" / "chunks.parquet"
INDEX_DIR = PROJECT_ROOT / "data" / "kb" / "chroma"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rag"
EVAL_QUERIES_PATH = PROJECT_ROOT / "data" / "kb" / "eval_queries.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(CHUNKS_PATH)
print(INDEX_DIR)

E:\Project\RadScribe
E:\Project\RadScribe\data\kb\processed\chunks.parquet
E:\Project\RadScribe\data\kb\chroma


In [15]:
chunks = pd.read_parquet(CHUNKS_PATH)

required_columns = {
    "chunk_id",
    "finding",
    "text",
    "source_ids",
    "source_titles",
    "source_urls",
    "licences",
}
missing = required_columns - set(chunks.columns)
assert not missing, f"Missing chunk columns: {missing}"
assert len(chunks) > 0, "No chunks found. Run 03_rag.ipynb first."

print("chunks:", len(chunks))
chunks.groupby("finding").size().to_frame("chunks")

chunks: 21


,chunks
finding,
Atelectasis,3
Cardiomegaly,4
Consolidation / Pneumonia,4
Edema,3
Pleural Effusion,4
Pneumothorax,3


## Evaluation Queries

Each query has one expected finding. These are not model-training labels; they are simple tests for the retriever.

In [16]:
eval_queries = [
    # Cardiomegaly
    {"query_id": "cardio_01", "query": "enlarged heart on chest x ray", "expected_finding": "Cardiomegaly", "scope": "in_scope"},
    {"query_id": "cardio_02", "query": "heart shadow is too large", "expected_finding": "Cardiomegaly", "scope": "in_scope"},
    {"query_id": "cardio_03", "query": "cardiothoracic ratio greater than fifty percent", "expected_finding": "Cardiomegaly", "scope": "in_scope"},
    {"query_id": "cardio_04", "query": "portable AP film makes the heart look enlarged", "expected_finding": "Cardiomegaly", "scope": "in_scope"},

    # Atelectasis
    {"query_id": "atel_01", "query": "volume loss in part of the lung", "expected_finding": "Atelectasis", "scope": "in_scope"},
    {"query_id": "atel_02", "query": "collapsed air spaces with elevated diaphragm", "expected_finding": "Atelectasis", "scope": "in_scope"},
    {"query_id": "atel_03", "query": "linear opacity from subsegmental collapse", "expected_finding": "Atelectasis", "scope": "in_scope"},
    {"query_id": "atel_04", "query": "loss of air and reduced lung volume", "expected_finding": "Atelectasis", "scope": "in_scope"},

    # Consolidation / Pneumonia
    {"query_id": "cons_01", "query": "air spaces filled with inflammatory material", "expected_finding": "Consolidation / Pneumonia", "scope": "in_scope"},
    {"query_id": "cons_02", "query": "pneumonia causing dense white opacity", "expected_finding": "Consolidation / Pneumonia", "scope": "in_scope"},
    {"query_id": "cons_03", "query": "air bronchograms inside an opacity", "expected_finding": "Consolidation / Pneumonia", "scope": "in_scope"},
    {"query_id": "cons_04", "query": "alveoli are filled instead of air filled", "expected_finding": "Consolidation / Pneumonia", "scope": "in_scope"},

    # Pleural Effusion
    {"query_id": "eff_01", "query": "fluid around the lung", "expected_finding": "Pleural Effusion", "scope": "in_scope"},
    {"query_id": "eff_02", "query": "blunting of the costophrenic angle", "expected_finding": "Pleural Effusion", "scope": "in_scope"},
    {"query_id": "eff_03", "query": "meniscus sign at the lung base", "expected_finding": "Pleural Effusion", "scope": "in_scope"},
    {"query_id": "eff_04", "query": "extra fluid in the pleural space", "expected_finding": "Pleural Effusion", "scope": "in_scope"},

    # Edema
    {"query_id": "edema_01", "query": "fluid buildup inside the lungs", "expected_finding": "Edema", "scope": "in_scope"},
    {"query_id": "edema_02", "query": "bat wing opacities and Kerley B lines", "expected_finding": "Edema", "scope": "in_scope"},
    {"query_id": "edema_03", "query": "vascular redistribution with pulmonary edema", "expected_finding": "Edema", "scope": "in_scope"},
    {"query_id": "edema_04", "query": "heart failure causing lung fluid", "expected_finding": "Edema", "scope": "in_scope"},

    # Pneumothorax
    {"query_id": "ptx_01", "query": "air in the pleural space", "expected_finding": "Pneumothorax", "scope": "in_scope"},
    {"query_id": "ptx_02", "query": "visible pleural line with no lung markings beyond it", "expected_finding": "Pneumothorax", "scope": "in_scope"},
    {"query_id": "ptx_03", "query": "collapsed lung from air outside the lung", "expected_finding": "Pneumothorax", "scope": "in_scope"},
    {"query_id": "ptx_04", "query": "tension pneumothorax is an emergency", "expected_finding": "Pneumothorax", "scope": "in_scope"},

    # Out-of-scope examples
    {"query_id": "oos_01", "query": "broken rib fracture after trauma", "expected_finding": "Out of scope", "scope": "out_of_scope"},
    {"query_id": "oos_02", "query": "kidney stone seen on CT abdomen", "expected_finding": "Out of scope", "scope": "out_of_scope"},
    {"query_id": "oos_03", "query": "brain MRI shows stroke", "expected_finding": "Out of scope", "scope": "out_of_scope"},
    {"query_id": "oos_04", "query": "patient has a skin rash on the arm", "expected_finding": "Out of scope", "scope": "out_of_scope"},
]

eval_df = pd.DataFrame(eval_queries)
eval_df.to_csv(EVAL_QUERIES_PATH, index=False)

print("queries:", len(eval_df))
print("saved:", EVAL_QUERIES_PATH)
eval_df.groupby(["scope", "expected_finding"]).size().to_frame("queries")

queries: 28
saved: E:\Project\RadScribe\data\kb\eval_queries.csv


queries
scope        expected_finding                  
in_scope     Atelectasis                      4
             Cardiomegaly                     4
             Consolidation / Pneumonia        4
             Edema                            4
             Pleural Effusion                 4
             Pneumothorax                     4
out_of_scope Out of scope                     4

## Load Retriever

Lower Chroma distance means a better match. For easier reading, this notebook also shows `display_score = 1 / (1 + distance)`, so higher is better.

In [17]:
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ModuleNotFoundError:
    pass

import os

api_key_present = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY present:", api_key_present)

OPENAI_API_KEY present: True


In [18]:
import chromadb
from openai import OpenAI

EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_COLLECTION = "radscribe_kb_openai"

assert api_key_present, "Set OPENAI_API_KEY before running this cell."

openai_client = OpenAI()
chroma_client = chromadb.PersistentClient(path=str(INDEX_DIR))
collection = chroma_client.get_collection(CHROMA_COLLECTION)

assert collection.count() == len(chunks), "Index chunk count does not match chunks.parquet. Re-run 03_rag.ipynb."

print("collection:", CHROMA_COLLECTION)
print("stored chunks:", collection.count())

collection: radscribe_kb_openai
stored chunks: 21


In [19]:
def embed_openai(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def retrieve(query: str, k: int = 5) -> list[dict]:
    query_embedding = embed_openai([query])[0]
    result = collection.query(query_embeddings=[query_embedding], n_results=k)

    rows = []
    for rank, i in enumerate(range(len(result["ids"][0])), start=1):
        distance = float(result["distances"][0][i])
        metadata = result["metadatas"][0][i]
        rows.append(
            {
                "rank": rank,
                "chunk_id": result["ids"][0][i],
                "finding": metadata["finding"],
                "text": result["documents"][0][i],
                "source_titles": metadata["source_titles"],
                "source_urls": metadata["source_urls"],
                "distance": distance,
                "display_score": 1 / (1 + distance),
            }
        )
    return rows


pd.DataFrame(retrieve("fluid around the lung", k=3))[["rank", "display_score", "finding", "chunk_id", "text"]]

,rank,display_score,finding,chunk_id,text
0,1,0.683558,Pleural Effusion,pleural_effusion_000,Pleural effusion means extra fluid has collect...
1,2,0.666315,Consolidation / Pneumonia,consolidation_pneumonia_002,It may look denser or whiter than normal aerat...
2,3,0.664485,Pleural Effusion,pleural_effusion_002,"On chest X-ray, pleural effusion is usually co..."


## Run Retrieval

In [20]:
all_hits = []

for query_row in eval_df.to_dict("records"):
    hits = retrieve(query_row["query"], k=5)
    for hit in hits:
        all_hits.append({**query_row, **hit})

hits_df = pd.DataFrame(all_hits)
hits_df.head(10)

,query_id,query,expected_finding,scope,rank,chunk_id,finding,text,source_titles,source_urls,distance,display_score
0,cardio_01,enlarged heart on chest x ray,Cardiomegaly,in_scope,1,cardiomegaly_001,Cardiomegaly,The finding matters because an enlarged cardia...,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.301457,0.768370
1,cardio_01,enlarged heart on chest x ray,Cardiomegaly,in_scope,2,cardiomegaly_002,Cardiomegaly,A common measurement is the cardiothoracic rat...,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.362275,0.734066
2,cardio_01,enlarged heart on chest x ray,Cardiomegaly,in_scope,3,cardiomegaly_000,Cardiomegaly,Cardiomegaly means the heart appears enlarged....,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.451801,0.688800
3,cardio_01,enlarged heart on chest x ray,Cardiomegaly,in_scope,4,edema_001,Edema,"Pulmonary edema can cause shortness of breath,...","[""Pulmonary edema""]","[""https://en.wikipedia.org/wiki/Pulmonary_edema""]",0.480072,0.675643
4,cardio_01,enlarged heart on chest x ray,Cardiomegaly,in_scope,5,pleural_effusion_002,Pleural Effusion,"On chest X-ray, pleural effusion is usually co...","[""Pleural Disorders - Pleurisy, Pleural Effusi...","[""https://www.nhlbi.nih.gov/health/pleural-dis...",0.504310,0.664757
5,cardio_02,heart shadow is too large,Cardiomegaly,in_scope,1,cardiomegaly_001,Cardiomegaly,The finding matters because an enlarged cardia...,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.663001,0.601323
6,cardio_02,heart shadow is too large,Cardiomegaly,in_scope,2,cardiomegaly_002,Cardiomegaly,A common measurement is the cardiothoracic rat...,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.687369,0.592639
7,cardio_02,heart shadow is too large,Cardiomegaly,in_scope,3,cardiomegaly_000,Cardiomegaly,Cardiomegaly means the heart appears enlarged....,"[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.743286,0.573629
8,cardio_02,heart shadow is too large,Cardiomegaly,in_scope,4,cardiomegaly_003,Cardiomegaly,"Follow-up with clinical history, prior imaging...","[""Heart Failure - What Is Heart Failure?"", ""Ca...","[""https://www.nhlbi.nih.gov/health/heart-failu...",0.763083,0.567188
9,cardio_02,heart shadow is too large,Cardiomegaly,in_scope,5,atelectasis_001,Atelectasis,"Atelectasis can happen for several reasons, in...","[""Atelectasis""]","[""https://en.wikipedia.org/wiki/Atelectasis""]",0.793222,0.557656


## Metrics

For this notebook:

- `recall@k` means: did the correct finding appear anywhere in the top k?
- `precision@k` means: what fraction of the top k hits were the correct finding?
- `MRR` means: how high was the first correct hit? Rank 1 is best.

In [21]:
def metrics_at_k(hits: pd.DataFrame, k: int) -> dict:
    in_scope = hits[hits["scope"] == "in_scope"].copy()
    per_query = []

    for query_id, group in in_scope.groupby("query_id"):
        group = group.sort_values("rank").head(k)
        expected = group["expected_finding"].iloc[0]
        matches = group["finding"].eq(expected).to_numpy()

        recall = float(matches.any())
        precision = float(matches.mean())
        reciprocal_rank = 0.0
        if matches.any():
            reciprocal_rank = 1.0 / float(np.where(matches)[0][0] + 1)

        per_query.append(
            {
                "query_id": query_id,
                "expected_finding": expected,
                f"recall@{k}": recall,
                f"precision@{k}": precision,
                f"rr@{k}": reciprocal_rank,
            }
        )

    per_query_df = pd.DataFrame(per_query)
    return {
        "k": k,
        f"recall@{k}": per_query_df[f"recall@{k}"].mean(),
        f"precision@{k}": per_query_df[f"precision@{k}"].mean(),
        f"mrr@{k}": per_query_df[f"rr@{k}"].mean(),
    }


metrics_df = pd.DataFrame([metrics_at_k(hits_df, k) for k in [1, 3, 5]])
metrics_df

,k,recall@1,precision@1,mrr@1,recall@3,precision@3,mrr@3,recall@5,precision@5,mrr@5
0,1,0.75,0.75,0.75,NaN,NaN,NaN,NaN,NaN,NaN
1,3,NaN,NaN,NaN,0.916667,0.611111,0.826389,NaN,NaN,NaN
2,5,NaN,NaN,NaN,NaN,NaN,NaN,0.958333,0.475,0.836806


In [22]:
per_query_rows = []

for query_id, group in hits_df[hits_df["scope"] == "in_scope"].groupby("query_id"):
    group = group.sort_values("rank")
    expected = group["expected_finding"].iloc[0]
    top_findings = group.head(5)["finding"].tolist()
    first_correct_rank = next((i + 1 for i, finding in enumerate(top_findings) if finding == expected), None)
    per_query_rows.append(
        {
            "query_id": query_id,
            "query": group["query"].iloc[0],
            "expected_finding": expected,
            "top1_finding": top_findings[0],
            "first_correct_rank": first_correct_rank,
            "top5_findings": top_findings,
        }
    )

per_query_df = pd.DataFrame(per_query_rows)
per_query_df.head(10)

,query_id,query,expected_finding,top1_finding,first_correct_rank,top5_findings
0,atel_01,volume loss in part of the lung,Atelectasis,Atelectasis,1.0,"[Atelectasis, Consolidation / Pneumonia, Atele..."
1,atel_02,collapsed air spaces with elevated diaphragm,Atelectasis,Atelectasis,1.0,"[Atelectasis, Atelectasis, Pneumothorax, Atele..."
2,atel_03,linear opacity from subsegmental collapse,Atelectasis,Consolidation / Pneumonia,4.0,"[Consolidation / Pneumonia, Consolidation / Pn..."
3,atel_04,loss of air and reduced lung volume,Atelectasis,Atelectasis,1.0,"[Atelectasis, Atelectasis, Consolidation / Pne..."
4,cardio_01,enlarged heart on chest x ray,Cardiomegaly,Cardiomegaly,1.0,"[Cardiomegaly, Cardiomegaly, Cardiomegaly, Ede..."
5,cardio_02,heart shadow is too large,Cardiomegaly,Cardiomegaly,1.0,"[Cardiomegaly, Cardiomegaly, Cardiomegaly, Car..."
6,cardio_03,cardiothoracic ratio greater than fifty percent,Cardiomegaly,Cardiomegaly,1.0,"[Cardiomegaly, Cardiomegaly, Cardiomegaly, Pne..."
7,cardio_04,portable AP film makes the heart look enlarged,Cardiomegaly,Cardiomegaly,1.0,"[Cardiomegaly, Cardiomegaly, Cardiomegaly, Car..."
8,cons_01,air spaces filled with inflammatory material,Consolidation / Pneumonia,Consolidation / Pneumonia,1.0,"[Consolidation / Pneumonia, Edema, Consolidati..."
9,cons_02,pneumonia causing dense white opacity,Consolidation / Pneumonia,Consolidation / Pneumonia,1.0,"[Consolidation / Pneumonia, Consolidation / Pn..."


In [23]:
per_finding_df = (
    per_query_df.assign(correct_at_1=per_query_df["first_correct_rank"].eq(1))
    .assign(correct_at_3=per_query_df["first_correct_rank"].le(3))
    .assign(correct_at_5=per_query_df["first_correct_rank"].le(5))
    .groupby("expected_finding")
    .agg(
        queries=("query_id", "count"),
        recall_at_1=("correct_at_1", "mean"),
        recall_at_3=("correct_at_3", "mean"),
        recall_at_5=("correct_at_5", "mean"),
        median_first_correct_rank=("first_correct_rank", "median"),
    )
)

per_finding_df

,queries,recall_at_1,recall_at_3,recall_at_5,median_first_correct_rank
expected_finding,,,,,
Atelectasis,4,0.75,0.75,1.00,1.0
Cardiomegaly,4,1.00,1.00,1.00,1.0
Consolidation / Pneumonia,4,0.75,1.00,1.00,1.0
Edema,4,0.50,0.75,0.75,1.0
Pleural Effusion,4,1.00,1.00,1.00,1.0
Pneumothorax,4,0.50,1.00,1.00,1.5


## Example Top-3 Hits

In [24]:
example_query_ids = ["eff_02", "ptx_02", "cons_03"]

example_hits = hits_df[hits_df["query_id"].isin(example_query_ids) & hits_df["rank"].le(3)].copy()
example_hits[["query_id", "query", "expected_finding", "rank", "finding", "display_score", "chunk_id", "text"]]

,query_id,query,expected_finding,rank,finding,display_score,chunk_id,text
50,cons_03,air bronchograms inside an opacity,Consolidation / Pneumonia,1,Consolidation / Pneumonia,0.752521,consolidation_pneumonia_002,It may look denser or whiter than normal aerat...
51,cons_03,air bronchograms inside an opacity,Consolidation / Pneumonia,2,Edema,0.676963,edema_002,Patchier alveolar opacities may be seen in non...
52,cons_03,air bronchograms inside an opacity,Consolidation / Pneumonia,3,Consolidation / Pneumonia,0.671468,consolidation_pneumonia_003,The report should describe the opacity pattern...
65,eff_02,blunting of the costophrenic angle,Pleural Effusion,1,Pleural Effusion,0.625670,pleural_effusion_002,"On chest X-ray, pleural effusion is usually co..."
66,eff_02,blunting of the costophrenic angle,Pleural Effusion,2,Atelectasis,0.625656,atelectasis_002,"Depending on the location, this may include cr..."
67,eff_02,blunting of the costophrenic angle,Pleural Effusion,3,Pneumothorax,0.612953,pneumothorax_002,Tension physiology may be suggested by mediast...
105,ptx_02,visible pleural line with no lung markings bey...,Pneumothorax,1,Pneumothorax,0.668566,pneumothorax_001,A tension pneumothorax is especially important...
106,ptx_02,visible pleural line with no lung markings bey...,Pneumothorax,2,Consolidation / Pneumonia,0.665148,consolidation_pneumonia_002,It may look denser or whiter than normal aerat...
107,ptx_02,visible pleural line with no lung markings bey...,Pneumothorax,3,Pleural Effusion,0.659788,pleural_effusion_002,"On chest X-ray, pleural effusion is usually co..."


## Out-of-Scope Check

These queries are not part of the six findings. We do not expect a perfect answer. We mainly want their top score to be lower than strong in-scope matches.

In [25]:
top_hits = hits_df[hits_df["rank"] == 1].copy()

score_summary = (
    top_hits.groupby("scope")["display_score"]
    .agg(["count", "mean", "min", "max"])
    .rename_axis("query_scope")
)

oos_top_hits = top_hits[top_hits["scope"] == "out_of_scope"][[
    "query_id", "query", "finding", "display_score", "chunk_id"
]]

display(score_summary)
oos_top_hits

,count,mean,min,max
query_scope,,,,
in_scope,24,0.688848,0.601323,0.788887
out_of_scope,4,0.586845,0.569839,0.605591


,query_id,query,finding,display_score,chunk_id
120,oos_01,broken rib fracture after trauma,Pneumothorax,0.605591,pneumothorax_001
125,oos_02,kidney stone seen on CT abdomen,Cardiomegaly,0.592735,cardiomegaly_002
130,oos_03,brain MRI shows stroke,Cardiomegaly,0.579213,cardiomegaly_003
135,oos_04,patient has a skin rash on the arm,Edema,0.569839,edema_002


## Save Evaluation Outputs

In [26]:
metrics_path = OUTPUT_DIR / "retrieval_metrics.csv"
per_query_path = OUTPUT_DIR / "retrieval_per_query.csv"
hits_path = OUTPUT_DIR / "retrieval_hits.csv"

metrics_df.to_csv(metrics_path, index=False)
per_query_df.to_csv(per_query_path, index=False)
hits_df.to_csv(hits_path, index=False)

print("saved:", metrics_path)
print("saved:", per_query_path)
print("saved:", hits_path)

saved: E:\Project\RadScribe\outputs\rag\retrieval_metrics.csv
saved: E:\Project\RadScribe\outputs\rag\retrieval_per_query.csv
saved: E:\Project\RadScribe\outputs\rag\retrieval_hits.csv


## Short Takeaway

Fill this in after running the notebook:

- Retrieval used OpenAI `text-embedding-3-small` over 21 KB chunks from six findings.
- On the in-scope eval set, recall@3 was `___` and MRR@3 was `___`.
- Out-of-scope questions had lower top scores than clear in-scope questions: `___`.
- Main failure pattern: `___`.

Retrieval used OpenAI text-embedding-3-small over 21 KB chunks from six findings.

On 24 in-scope eval queries, recall@3 was 0.917 and MRR@3 was 0.826.

Out-of-scope questions had lower top scores than in-scope questions: 0.415 vs 0.528.

Main failure pattern: edema and pneumothorax can be confused with pleural effusion when the query uses broad words like “fluid” or “pleural space.”